# Berlin AQI — Exploratory Data Analysis

Phase 1.1 of the planning doc. Goal: understand the data before writing any pipeline code.

**Seven questions to answer:**

1. How many Berlin stations are there, and what are their location IDs?
2. Which stations report PM2.5?
3. Which also report temperature and relative humidity?
4. How far back does each station's data go? (need ≥2 years)
5. How complete is the data per parameter per station?
6. What does the PM2.5 distribution look like? Class imbalance across risk categories?
7. Should SO₂, CO, and BC be included? (check variance)

We start with Berlin Mitte (`location_id=3019`) to validate the API shape, then expand.

## Setup

In [ ]:
import os
import sys
from pathlib import Path

import httpx
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv

# make src/ importable
sys.path.insert(0, str(Path.cwd().parent))

load_dotenv(Path.cwd().parent / ".env")

API_KEY = os.getenv("OPENAQ_API_KEY")
assert API_KEY, "OPENAQ_API_KEY not set in .env"

BASE = "https://api.openaq.org/v3"
HEADERS = {"X-API-Key": API_KEY}
# Berlin bounding box: minLon, minLat, maxLon, maxLat
BERLIN_BBOX = "13.088,52.338,13.761,52.675"
BERLIN_MITTE_ID = 3019

pd.set_option("display.max_rows", 60)
pd.set_option("display.width", 160)

## Q1 — Berlin stations

Pull every location inside the Berlin bounding box. The planning doc expects ~21.

In [ ]:
with httpx.Client(base_url=BASE, headers=HEADERS, timeout=30.0) as c:
    r = c.get("/locations", params={"bbox": BERLIN_BBOX, "limit": 1000})
    r.raise_for_status()
    locations = r.json()["results"]

print(f"Found {len(locations)} stations in Berlin bbox")

stations_df = pd.DataFrame(
    [
        {
            "id": l["id"],
            "name": l["name"],
            "provider": (l.get("provider") or {}).get("name"),
            "n_sensors": len(l.get("sensors", []) or []),
        }
        for l in locations
    ]
).sort_values("id")
stations_df

## Q2 + Q3 — Parameter coverage

Which stations report PM2.5, temperature, and relative humidity? Build a station × parameter coverage matrix.

In [ ]:
sensor_rows = []
for l in locations:
    for s in l.get("sensors", []) or []:
        p = s.get("parameter") or {}
        sensor_rows.append(
            {
                "id": l["id"],
                "name": l["name"],
                "sensor_id": s["id"],
                "parameter": p.get("name"),
                "units": p.get("units"),
            }
        )

sensors_df = pd.DataFrame(sensor_rows)
print("Distinct parameters seen:", sorted(sensors_df["parameter"].dropna().unique()))

coverage = (
    sensors_df.assign(has=1)
    .pivot_table(index=["id", "name"], columns="parameter", values="has", fill_value=0)
    .astype(int)
)
coverage

## API shape check — Berlin Mitte (3019)

Before fetching 2 years of data, verify the response structure on a single station.

In [ ]:
with httpx.Client(base_url=BASE, headers=HEADERS, timeout=30.0) as c:
    r = c.get(f"/locations/{BERLIN_MITTE_ID}")
    r.raise_for_status()
    mitte = r.json()["results"][0]

print("Name:    ", mitte["name"])
print("Country: ", (mitte.get("country") or {}).get("code"))
print("Timezone:", mitte.get("timezone"))
print("Provider:", (mitte.get("provider") or {}).get("name"))
print(f"Sensors ({len(mitte['sensors'])}):")
for s in mitte["sensors"]:
    p = s.get("parameter", {}) or {}
    first = (s.get("datetimeFirst") or {}).get("utc")
    last = (s.get("datetimeLast") or {}).get("utc")
    print(f"  - {p.get('name'):18s} ({p.get('units'):8s}) id={s['id']:<7} first={first} last={last}")

In [ ]:
# Peek at a single hour record to see what fields we get back
pm25_sensor = next(
    s for s in mitte["sensors"] if (s.get("parameter") or {}).get("name") == "pm25"
)

with httpx.Client(base_url=BASE, headers=HEADERS, timeout=30.0) as c:
    r = c.get(f"/sensors/{pm25_sensor['id']}/hours", params={"limit": 1})
    r.raise_for_status()
    sample = r.json()["results"][0]

import json
print(json.dumps(sample, indent=2, default=str))

## Q4 — Time coverage per station

How many years of PM2.5 data does each station have? Need ≥2 years for training.

In [ ]:
time_rows = []
for l in locations:
    for s in l.get("sensors", []) or []:
        if (s.get("parameter") or {}).get("name") != "pm25":
            continue
        first = (s.get("datetimeFirst") or {}).get("utc")
        last = (s.get("datetimeLast") or {}).get("utc")
        time_rows.append(
            {
                "id": l["id"],
                "name": l["name"],
                "sensor_id": s["id"],
                "first": first,
                "last": last,
            }
        )

time_df = pd.DataFrame(time_rows)
time_df["first"] = pd.to_datetime(time_df["first"], utc=True, errors="coerce")
time_df["last"] = pd.to_datetime(time_df["last"], utc=True, errors="coerce")
time_df["years"] = ((time_df["last"] - time_df["first"]).dt.days / 365.25).round(2)
time_df.sort_values("years", ascending=False)

## Q5 + Q6 — Deep dive on Berlin Mitte

Fetch 2 years of hourly data for station 3019 (via `src.ingest.fetch`, which already handles pagination).
Then answer:
- **Q5:** NaN % per parameter
- **Q6:** PM2.5 distribution and class balance across the custom AQI categories

In [ ]:
from src.ingest import fetch

now = pd.Timestamp.now(tz="UTC")
iso = "%Y-%m-%dT%H:%M:%SZ"
datetime_from = (now - pd.Timedelta(days=730)).strftime(iso)
datetime_to = now.strftime(iso)

raw = fetch(BERLIN_MITTE_ID, datetime_from, datetime_to)
print(f"Rows: {len(raw):,}")
print(f"Date range: {raw['datetime'].min()} → {raw['datetime'].max()}")
raw.head()

In [ ]:
# Q5 — NaN percentage per column
nan_pct = (raw.isna().mean() * 100).round(2).sort_values(ascending=False)
nan_pct.to_frame("nan_pct")

In [ ]:
# Q6a — PM2.5 distribution
fig, ax = plt.subplots(figsize=(10, 4))
raw["pm25"].dropna().clip(upper=150).hist(bins=60, ax=ax)
ax.set_xlabel("PM2.5 (µg/m³)  — clipped at 150 for readability")
ax.set_ylabel("Hours")
ax.set_title(f"PM2.5 hourly distribution — Berlin Mitte ({BERLIN_MITTE_ID})")
plt.tight_layout()
plt.show()

raw["pm25"].describe().round(2)

In [ ]:
# Q6b — Class balance across custom AQI categories
AQI_BINS = [0, 12, 35.4, 55.4, 150.4, 250.4, float("inf")]
AQI_LABELS = ["All Clear", "Low Risk", "Elevated", "Significant", "High", "Very High"]

aqi = pd.cut(raw["pm25"].dropna(), bins=AQI_BINS, labels=AQI_LABELS, include_lowest=True)
balance = aqi.value_counts().reindex(AQI_LABELS).fillna(0).astype(int)
balance_pct = (balance / balance.sum() * 100).round(2)

pd.DataFrame({"count": balance, "pct": balance_pct})

## Q7 — Variance of SO₂, CO, BC

A pollutant that barely changes in Berlin has no predictive value. Compute coefficient of variation (std / mean) for every pollutant available at Berlin Mitte.

In [ ]:
pollutants = [c for c in ["pm25", "pm10", "no2", "o3", "so2", "co"] if c in raw.columns]
stats = raw[pollutants].describe().T
stats["cv"] = (stats["std"] / stats["mean"]).round(3)
stats[["count", "mean", "std", "cv"]]

## Summary — usable station list

A station is "usable" if it reports PM2.5 + temperature + relative humidity AND has ≥2 years of PM2.5 history.

In [ ]:
pm25_ids = set(sensors_df.loc[sensors_df["parameter"] == "pm25", "id"])
temp_ids = set(sensors_df.loc[sensors_df["parameter"] == "temperature", "id"])
rh_ids = set(sensors_df.loc[sensors_df["parameter"] == "relativehumidity", "id"])

print(f"Stations with PM2.5:              {len(pm25_ids)}")
print(f"Stations with temperature:        {len(temp_ids)}")
print(f"Stations with relative humidity:  {len(rh_ids)}")

has_all_three = pm25_ids & temp_ids & rh_ids
long_history = set(time_df.loc[time_df["years"] >= 2, "id"])
usable = sorted(has_all_three & long_history)

print(f"\nUsable stations (PM2.5 + temp + RH + ≥2y history): {len(usable)}")
print(usable)

stations_df[stations_df["id"].isin(usable)]